# Build your own scenario

A blank notebook to fill in. It runs as it stands, against
`blank_scenario.yaml` in this folder, so you can execute the whole thing once
to see what a working scenario looks like, then start replacing the numbers
with yours.

The order below is the order that saves time. Check the design point before
you commit to a time series: a scenario that fails at design will fail once
per snapshot, 336 times in a row.

1. Point at a scenario file
2. Look at the plan view
3. Check the design point
4. Edit the scenario, from Python or in the file
5. Run the time series
6. Read the results

If you would rather not leave the shell, the same steps are
`discoolpy plot`, `discoolpy check` and `discoolpy run`.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

import discoolpy
from discoolpy import (
    build_system,
    check_scenario,
    load_scenario,
    plot_network,
    run_scenario,
)

print(f"discoolpy {discoolpy.__version__}")

## 1. Point at a scenario file

`blank_scenario.yaml` sits next to this notebook. Open it in an editor
alongside: it is commented section by section, and every value marked TODO is
one you are meant to replace.

`load_scenario` gives you the file as an ordinary dictionary, so anything you
would rather set from Python you can set here instead of editing YAML.

In [ ]:
SCENARIO = Path("blank_scenario.yaml")

config = load_scenario(SCENARIO)
print(config["metadata"]["description"].strip())
print()
for b in config["buildings"]:
    print(f"  {b['label']:<14} {b['Q_design_W'] / 1e3:6.0f} kW   {b.get('archetype', 'office')}")

## 2. Look at the plan view

Nothing has been solved yet. The layout comes from the pipe lengths and
bearings in the scenario, so it is worth drawing first: a spur pointing the
wrong way or a mislabelled branch is cheap to fix now and annoying to find
later.

The template starts with `autofill: true` and no pipe geometry, so the drawing
you get is schematic. Give the pipes real `L` and `heading_deg` values and it
becomes a plan you can hold against a site drawing.

In [ ]:
system = build_system(config, strict_hydraulics=False)
ax = plot_network(system, annotate_pipes=True)
ax.figure.set_size_inches(9, 7)
plt.show()

## 3. Check the design point

This solves once, at design, and reports the things that usually go wrong:
the branch tree, the pressure degrees of freedom, the pipe heat-gain
conductances, the plant energy balance, and whether the pump can actually
serve the far end of the network.

Read the energy balance line by line. The residual has to be about zero, and
the plant duty should sit above the sum of the building loads by the pipe gain
and the pump heat.

In [ ]:
result = check_scenario(SCENARIO)
print()
print("design point OK" if result.ok else "design point FAILED, see above")

## 4. Edit the scenario

Two ways, and they mix freely.

**In the file.** Open `blank_scenario.yaml`, change what you want, and re-run
the cells above. This is the one to use for anything you want to keep.

**From Python.** The dictionary you loaded is yours to modify, and every
function here takes a dictionary as happily as a path. Good for a sweep, or
for trying something before you commit it to the file.

The cell below is a worked example of the second: it gives the network real
pipe geometry, turns on buried-pipe heat gain, and adds a third building. Edit
it, or delete it and write your own.

In [ ]:
config = load_scenario(SCENARIO)

# A third building on the same street.
config["buildings"].append(
    {"label": "building_3", "Q_design_W": 200_000.0, "archetype": "retail"}
)

# Real geometry, so pressure drop and heat gain both come from the pipes
# rather than from a nominal pressure ratio.
config["branch"]["autofill"] = False
config["branch"]["pipe_model"] = "darcy"
config["branch"]["heat_model"] = "ua"
config["branch"]["fix_pump_power"] = False
config["branch"]["pump_pressure_ratio"] = 1.25
config["branch"]["thermal_defaults"] = {
    "placement": "buried",
    "insulation": "pur",
    "insulation_thickness_m": 0.05,
    "burial_depth_m": 1.2,
    "ground": "moist soil",
    "twin_spacing_m": 0.7,
    "ambient_source": "ground",
}
config["branch"]["pipes"] = {
    "supply_1": {"L": 400.0, "D": 0.15, "ks": 5e-5, "heading_deg": 0},
    "supply_2": {"L": 250.0, "D": 0.125, "ks": 5e-5, "heading_deg": -30},
    "supply_3": {"L": 200.0, "D": 0.10, "ks": 5e-5, "heading_deg": 20},
    "return_3": {"L": 200.0, "D": 0.10, "ks": 5e-5},
    "return_2": {"L": 250.0, "D": 0.125, "ks": 5e-5},
    "return_1": {"L": 400.0, "D": 0.15, "ks": 5e-5},
}

checked = check_scenario(config)

Buried pipes now gain heat from the soil, so the plant duty in that report is
above the sum of the building loads. That gap is real: it is cooling the plant
has to make and nobody gets to use.

Draw it again and the network has a shape, because the pipes now carry lengths
and bearings.

In [ ]:
ax = plot_network(build_system(config), annotate_pipes=True)
ax.figure.set_size_inches(10, 7)
plt.show()

## 5. Run the time series

`run_scenario` does the rest: generate the profile, solve the design point,
step through every snapshot, write the results out.

Start short. `periods=48` is one day at half-hourly resolution and takes under
a minute; the week the scenario asks for takes seven times that. Raise it once
the numbers look sensible.

If the scenario enables a store, this runs twice by default, once without it
and once with it, and assesses the difference. That comparison is what a
storage study is actually after.

In [ ]:
run = run_scenario(config, periods=48)
print()
print(run.summary())

In [ ]:
if run.report is not None:
    print(run.report.to_markdown())
else:
    print("No store enabled, so there is nothing to compare.")
    print("Set storage.enabled: true in the scenario to get a flexibility assessment.")

## 6. Read the results

One row per snapshot, one column per quantity. The columns you will reach for
first:

| Column | Meaning |
|---|---|
| `actual_building_total_Q_W` | What the buildings consumed |
| `chiller_Q_evap_W` | What the plant had to produce |
| `pipe_heat_gain_W` | The difference the distribution network is responsible for |
| `compressor_power_W` | Electrical demand, the thing you are trying to shift |
| `cop` | Plant efficiency at that snapshot |
| `chw_supply_T_degC`, `chw_return_T_degC` | Distribution temperatures |
| `storage_power_W`, `storage_soc_after` | Store dispatch, where there is one |

Every column is in `docs/configuration.md` under Result columns.

In [ ]:
frame = run.results
columns = [c for c in [
    "timestamp",
    "actual_building_total_Q_W",
    "chiller_Q_evap_W",
    "pipe_heat_gain_W",
    "compressor_power_W",
    "cop",
] if c in frame.columns]
frame[columns].head(8)

In [ ]:
frame = run.results.copy()
frame["timestamp"] = pd.to_datetime(frame["timestamp"])

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].plot(frame["timestamp"], frame["actual_building_total_Q_W"] / 1e3,
             color="black", label="Building demand")
axes[0].plot(frame["timestamp"], frame["chiller_Q_evap_W"] / 1e3,
             color="tab:blue", label="Plant duty")
axes[0].set_ylabel("Cooling [kW]")
axes[0].set_title("The gap between the two is pipe gain, pump heat and any store")
axes[0].legend(loc="upper left")

axes[1].plot(frame["timestamp"], frame["compressor_power_W"] / 1e3, color="tab:red")
axes[1].set_ylabel("Compressor power [kW]")
axes[1].set_xlabel("Time")

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Where to go next

- `docs/configuration.md` is the full key list, section by section.
- `docs/network_topology.md` covers forks, satellite plants, and the degree of
  freedom rules that decide which elements you may pin.
- `docs/heat_gains.md` explains where the conductances come from.
- The shipped scenarios in `configs/` are each built around one question;
  `discoolpy list` prints what each is for.

Some things worth trying on your own scenario, roughly in order of how much
they change the answer:

- Turn a pipe from `adiabatic` to `ua` and watch the plant duty rise above the
  building demand.
- Add a store, set `target_chiller_load_kW` near the mean plant duty, and read
  the electric round-trip efficiency rather than the thermal one.
- Give a building `load_model: thermal_mass` and a comfort band, then pre-cool
  it. Check `<building>_band_violation_K` afterwards: flexibility taken out of
  occupant comfort is not flexibility.
- Split the street with a `type: branch` terminal and hang a satellite plant
  off one arm.